In [116]:
import numpy as np
import astropy 
from astropy.io import fits, ascii 
from astropy.table import Table, Column
import subprocess
import os
import glob
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

In [2]:
# read in files
path = '../data/'
hdu = fits.open(path + "imbh_sample.fits")
imbh = hdu[1].data
hdu = fits.open(path+'spAll-v6_1_3.fits')
dr19_download = hdu[1].data

In [17]:
sp = ascii.read('./sdss_softening_param.txt')
zcutoff=0.7
def sdss_flux_mag(f, band): 
    idx = np.where(sp['filter']=='i')[0][0]
    b = sp['b'][idx]
    return -2.5/np.log10(10) * (np.asinh((f/1E9)/(2*b)) + np.log10(b))
def lrg_search(source):
    sdss_i = sdss_flux_mag(source['SDSS_CALIBFLUX_i'], 'i')
    sdss_z = sdss_flux_mag(source['SDSS_CALIBFLUX_z'], 'z')
    sdss_r = sdss_flux_mag(source['SDSS_CALIBFLUX_r'], 'r')
    # definitions from SDSS DR17 
    lrg_izw = (sdss_i - sdss_z > 0.7) & (sdss_i - source['WISE_w1mpro'] > (2.143)*(sdss_i -sdss_z) - 0.2) & (sdss_z < 19.95) & (sdss_i > 19.9)
    lrg_riw = (sdss_r - sdss_i > 0.98) & (sdss_r - source['WISE_w1mpro'] > 2*(sdss_r - sdss_i)) & (sdss_i - sdss_z > 0.625) & (sdss_z < 19.95) & (sdss_i > 19.9)
    lrg = (lrg_izw) | (lrg_riw)
    return(lrg)

In [20]:
for source in imbh: 
    if(np.isnan(source['DESI_TARGETID'])): 
        if(source['dr'] == 17): 
            survey = 'dr17'
        elif(source['dr'] == 19): 
            survey = 'dr19'
        else: 
            survey = None
    else: 
        survey = 'desi'
    highz = bool(source['combo_Z'] >= zcutoff)
    if(highz): 
        lrg = lrg_search(source)
        if(lrg): 
            fit_continuum = True 
        else: 
            fit_continuum = False 
    else: 
        fit_continuum = True 
    ## find file 
    if(survey == 'dr17'): 
        file = f"spec-{int(source['plate']):04d}-{int(source['mjd']):05d}-{int(source['fiberid']):04d}.fits"
    if(survey == 'dr19'): 
        specobjid = source['SDSS_SPECOBJID'].split("'")[1].strip()
        m = (specobjid == dr19_download['SPECOBJID'])
        idx = np.where(m == True)
        file = dr19_download['SPEC_FILE'][idx][0]
    if(survey == 'dr17' or survey == 'dr19'): 
        path = '/Users/f007znp/Research/processed/SDSS_BOSS/'
    if(fit_continuum): 
        if(survey == 'dr19'): 
            with open("fit_continuum.txt", 'a') as f: 
                f.write(f"{path}{file} {source['field']}\n")
        elif(survey == 'dr17'): 
            with open("fit_continuum.txt", 'a') as f: 
                f.write(f"{path}{file} {source['plate']}\n")
    elif(~fit_continuum): 
        if(survey == 'dr19'): 
            with open("no_continuum.txt", 'a') as f: 
                f.write(f"{path}{file} {source['field']}\n")
        elif(survey == 'dr17'): 
            with open("no_continuum.txt", 'a') as f: 
                f.write(f"{path}{file} {source['plate']}\n")

/var/folders/m1/ly87n3sn4g31x85g5tlkr4b80000gp/T/ipykernel_63766/2553811192.py:37: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  elif(~fit_continuum):


Run in gdl 
<code>process_survey,'sdss',inptable='../IMBH/nburst_fittingscript/no_continuum.txt',/plot,nlosvd=3,emexcl=0,emlt1=[1],emlt2=[2],moments=4,start=[0,100,0,80,0,400,3000,-1.2],siglimits=[500,500,3000],/disable_stpop,/xsl,lammin=3700,lammax=9000,degree=2,mdegree=5,path_ssp='/Users/f007znp/Research/stellar_templates/XSL/Kroupa/',prefix='SB_',suffix='_XSL_Kroupa_PC.fits',outpath='SDSS_BOSS/scripttest/'</code>


<code>process_survey,'sdss',inptable='../IMBH/nburst_fittingscript/fit_continuum.txt',/plot,nlosvd=3,emexcl=0,emlt1=[1],emlt2=[2],moments=4,start=[0,100,0,80,0,400,3000,-1.2],siglimits=[500,500,3000],/xsl,lammin=3700,lammax=9000,degree=2,mdegree=5,path_ssp='/Users/f007znp/Research/stellar_templates/XSL/Kroupa/',prefix='SB_',suffix='_XSL_Kroupa_PC.fits',outpath='SDSS_BOSS/scripttest/'</code>

## Inspect NL + BL fits 

In [78]:
def compute_rms(target, window): 
    idx = np.where(hdu[2].data["LINE_ID"][0] == target)[0][0]
    wavelength = hdu[2].data['WAVE'][0][idx]
    spec = hdu[1].data 
    wave = spec['WAVE'][0]
    flux = spec['FLUX'][0]
    fit_total = spec['FIT'][0]
    error = spec['ERROR'][0]
    fit_comp = spec['FIT_COMP'][0]
    broad = fit_comp[2]
    narrow = fit_total - broad 
    mask = (wave > wavelength - window) & (wave < wavelength + window)
    resid_with_broad = np.sqrt(np.mean(((flux[mask] - fit_total[mask]) / error[mask])**2))
    resid_witho_broad = np.sqrt(np.mean(((flux[mask] - narrow[mask]) / error[mask])**2))
    return resid_with_broad, resid_witho_broad
def classify(resid_with_broad, resid_witho_broad): 
    if(np.abs(1-resid_with_broad)< np.abs(1-resid_witho_broad)): 
        return "broad"
    else: 
        return "narrow"

In [147]:
path = '../../pro/SDSS_BOSS/scripttest/'
window = 80
folders = glob.glob(path+"/*")
filelist=[]
classification = []
poor_fit_flag = []
for folder in folders: 
    files = glob.glob(folder + "/*")
    for file in files:
        filelist.append(file)
        hdu= fits.open(file)
        lammin = hdu[0].header['LAMMIN']
        lammax = hdu[0].header['LAMMAX']
        if(lammax > 6562.8 and lammin< 6562.8): 
            resid_with_broad, resid_witho_broad = compute_rms("H alpha", window)
            classification.append(classify(resid_with_broad, resid_witho_broad))
            if(((resid_with_broad >=1.5) or (resid_with_broad <= 0.5)) and ((resid_witho_broad >=1.5) or (resid_witho_broad <= 0.5)) or (hdu[0].header['CHI2'] < 0.5) or (hdu[0].header['CHI2'] > 1.5)): 
                poor_fit_flag.append(True)
            else: 
                poor_fit_flag.append(False)
        else: 
            resid_with_broad, resid_witho_broad = compute_rms("Mg II] 2796", window)
            mg_class =classify(resid_with_broad, resid_witho_broad)
            resid_with_broad, resid_witho_broad = compute_rms("H beta", window)
            hbeta_class = classify(resid_with_broad, resid_witho_broad)
            if(mg_class == "narrow" and hbeta_class == "narrow"): 
                classification.append("narrow")
            else: 
                classification.append("broad")
            if(((resid_with_broad >=1.5) or (resid_with_broad <= 0.5)) and ((resid_witho_broad >=1.5) or (resid_witho_broad <= 0.5)) or (hdu[0].header['CHI2'] < 0.5) or (hdu[0].header['CHI2'] > 1.5)): 
                poor_fit_flag.append(True)
            else: 
                poor_fit_flag.append(False)
        

In [148]:
round1_output = {
    "file": filelist, 
    "classification": classification, 
    "fit flag": poor_fit_flag
}
round1_output = Table(round1_output)
ascii.write(round1_output, 'round1_output.txt', format = 'csv', overwrite = True)